In [7]:
import pandas as pd
import numpy as np

In [8]:
df=pd.read_csv(r'C:\Users\Karanpreet singh\OneDrive\Desktop\influenceguard_ai\data\processed.csv')
df.head(5)

,username,followers,post_frequency,likes,comments,shares,platform,verified,engagement_rate,is_fake
0,user_0,122958,25,9557,263,296,Instagram,0,0.0823,0
1,user_1,147867,22,20854,1242,416,Instagram,0,0.1523,0
2,user_2,132932,27,7246,215,73,Instagram,0,0.0567,0
3,user_3,366838,27,22357,789,379,Instagram,0,0.0641,0
4,user_4,260178,17,23198,1781,340,Instagram,0,0.0973,0


In [9]:
df["engagement_rate"] = (
    df["likes"] + df["comments"] + df["shares"]
) / df["followers"]

df["like_ratio"] = df["likes"] / df["followers"]
df["comment_ratio"] = df["comments"] / df["followers"]
df["share_ratio"] = df["shares"] / df["followers"]

In [10]:
df.head(5)

,username,followers,post_frequency,likes,comments,shares,platform,verified,engagement_rate,is_fake,like_ratio,comment_ratio,share_ratio
0,user_0,122958,25,9557,263,296,Instagram,0,0.082272,0,0.077726,0.002139,0.002407
1,user_1,147867,22,20854,1242,416,Instagram,0,0.152245,0,0.141032,0.008399,0.002813
2,user_2,132932,27,7246,215,73,Instagram,0,0.056676,0,0.054509,0.001617,0.000549
3,user_3,366838,27,22357,789,379,Instagram,0,0.064129,0,0.060945,0.002151,0.001033
4,user_4,260178,17,23198,1781,340,Instagram,0,0.097314,0,0.089162,0.006845,0.001307


Behavioral Features

In [11]:
df["like_comment_ratio"] = df["likes"] / (df["comments"] + 1)
df["activity_rate"] = df["post_frequency"]

Platform Weighting

In [12]:
platform_weights = {
    "Instagram": 1.0,
    "Twitter": 0.7,
    "Facebook": 0.85
}

df["platform_weight"] = df["platform"].map(platform_weights).fillna(0.8)

Platform Adjusted Engagement

In [13]:
df["platform_engagement"] = df["engagement_rate"] * df["platform_weight"]

Data-Driven Flags

In [14]:
#low enagement 
low_threshold = df["engagement_rate"].quantile(0.25)

df["low_engagement_flag"] = (
    df["engagement_rate"] < low_threshold
).astype(int)

In [15]:
#Viral Content
viral_threshold = df["engagement_rate"].quantile(0.90)

df["is_viral"] = (
    df["engagement_rate"] > viral_threshold
).astype(int)

Fake Influencer Indicator

In [16]:
df["fake_flag"] = np.where(
    (df["followers"] > df["followers"].quantile(0.75)) &
    (df["engagement_rate"] < low_threshold),
    1, 0
)

Engagement Score

In [17]:
df["engagement_score"] = (
    df["like_ratio"] +
    2 * df["comment_ratio"] +
    3 * df["share_ratio"]
)

In [18]:
df.head()

,username,followers,post_frequency,likes,comments,shares,platform,verified,engagement_rate,is_fake,...,comment_ratio,share_ratio,like_comment_ratio,activity_rate,platform_weight,platform_engagement,low_engagement_flag,is_viral,fake_flag,engagement_score
0,user_0,122958,25,9557,263,296,Instagram,0,0.082272,0,...,0.002139,0.002407,36.200758,25,1.0,0.082272,0,0,0,0.089226
1,user_1,147867,22,20854,1242,416,Instagram,0,0.152245,0,...,0.008399,0.002813,16.777152,22,1.0,0.152245,0,1,0,0.166271
2,user_2,132932,27,7246,215,73,Instagram,0,0.056676,0,...,0.001617,0.000549,33.546296,27,1.0,0.056676,0,0,0,0.059391
3,user_3,366838,27,22357,789,379,Instagram,0,0.064129,0,...,0.002151,0.001033,28.300000,27,1.0,0.064129,0,0,0,0.068346
4,user_4,260178,17,23198,1781,340,Instagram,0,0.097314,0,...,0.006845,0.001307,13.017957,17,1.0,0.097314,0,0,0,0.106773


In [21]:
df.drop('activity_rate',axis=1,inplace=True)

In [22]:
df.head()

,username,followers,post_frequency,likes,comments,shares,platform,verified,engagement_rate,is_fake,like_ratio,comment_ratio,share_ratio,like_comment_ratio,platform_weight,platform_engagement,low_engagement_flag,is_viral,fake_flag,engagement_score
0,user_0,122958,25,9557,263,296,Instagram,0,0.082272,0,0.077726,0.002139,0.002407,36.200758,1.0,0.082272,0,0,0,0.089226
1,user_1,147867,22,20854,1242,416,Instagram,0,0.152245,0,0.141032,0.008399,0.002813,16.777152,1.0,0.152245,0,1,0,0.166271
2,user_2,132932,27,7246,215,73,Instagram,0,0.056676,0,0.054509,0.001617,0.000549,33.546296,1.0,0.056676,0,0,0,0.059391
3,user_3,366838,27,22357,789,379,Instagram,0,0.064129,0,0.060945,0.002151,0.001033,28.300000,1.0,0.064129,0,0,0,0.068346
4,user_4,260178,17,23198,1781,340,Instagram,0,0.097314,0,0.089162,0.006845,0.001307,13.017957,1.0,0.097314,0,0,0,0.106773


In [23]:
import os

base_path = r"C:/Users/Karanpreet singh/OneDrive/Desktop/influenceguard_ai/data"

os.makedirs(base_path, exist_ok=True)

df.to_csv(os.path.join(base_path, "feature_engineered_data.csv"), index=False)